# 01 — Rolando: Data Ingestion & Annotation-Aware Manifest

**Google Colab notebook.** Runtime → *Change runtime type* → **GPU (T4 / L4 / A100)** before running.

Builds the invoice manifest that every other stage depends on. **This version samples annotation-aware**, so the manifest carries OCR/field ground truth instead of the 26% coverage the first local run produced.

| | |
|---|---|
| **Inputs** | Drive `inputs/datasets/invoices_raw/` (optional — see note) |
| **Outputs** | `invoice_manifest.csv`, QA report, 2 figures |
| **Expected runtime** | ~10–20 min (optional — the manifest was already built locally) |
| **Compute profile** | `colab_gpu` (generous — full data, pinned in the profile cell) |

### How results get back to the team
Everything is written to Google Drive by `colab_bootstrap.publish()`, into **both**:
- `outputs/rolando/<kind>/` — the *latest* copy
- `runs/rolando/<UTC-timestamp>/<kind>/` — an immutable archive, so re-running never
  silently destroys an earlier result

Tell the integrator (Hessam) when you're done; he copies from `outputs/` into the repo.

> **Before you run:** `MyDrive/DL2_InvoiceAI/` must already contain `code/` (the repo's `src/`,
> `scripts/`, and `colab_bootstrap.py`) and `inputs/`. If it doesn't, the bootstrap cell fails
> fast with a message telling you exactly what's missing.

### Why this notebook exists

The first manifest stratified across all three batches for *visual diversity*, blind to labels.
But annotation CSVs exist **only for batch_1** — so only **197 of 750 images (26.3%)** had ground
truth, and just 26 in the test split. That is too thin to report an OCR or field-extraction score on.

Two corrections here:
1. **Annotation-aware sampling** — prefer images that have a row in the batch_1 annotation CSVs,
   lifting GT coverage toward ~100%.
2. **Skip the duplicates** — `batch_3/` contains full copies of `batch_1/` and `batch_2/` beside
   its own `batch3_*` folders. True unique images = **5,201**, not the 8,181 you get by counting
   everything. Sampling the duplicates would put the same invoice in train *and* test.

In [ ]:
# --- GPU check: stop here if this says "no GPU" ---------------------------------
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or
      "!! NO GPU. Runtime > Change runtime type > Hardware accelerator = GPU, then re-run.")
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
# --- Mount Drive + load the shared bootstrap -----------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/DL2_InvoiceAI"   # <-- change if your folder differs

import sys, os, shutil, json, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

_bs = Path(DRIVE_ROOT) / "code" / "colab_bootstrap.py"
assert _bs.exists(), (
    f"Missing {_bs}.\nUpload the repo's colab/colab_bootstrap.py into "
    f"{DRIVE_ROOT}/code/ and re-run this cell."
)
sys.path.insert(0, str(_bs.parent))
import colab_bootstrap as CB

root  = CB.mount_drive(DRIVE_ROOT)
paths = CB.setup_paths(root)
CB.install_deps("pandas", "pillow", "tqdm")
print("Drive root:", root)

In [ ]:
# --- Pin the generous Colab budget --------------------------------------------
os.environ["IIP_COMPUTE_PROFILE"] = "colab_gpu"
from src.compute_profile import get_profile

P = get_profile()
print(json.dumps(P, indent=2, default=str))

RUN_TS = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())   # one archive folder for this run
T0 = time.time()

In [ ]:
# --- Datasets: read straight from Google Drive (NO Kaggle token needed) ------
# OPTIONAL stage: the manifest in inputs/ was already produced from this data locally. You only need invoices_raw/ in Drive if you want to re-derive the manifest yourself.
# Paths are resolved tolerantly: if a dataset was copied one level too deep
# (e.g. ocr_multitype/invoice/train/... instead of ocr_multitype/train/...),
# it is found anyway and a NOTE is printed. No re-upload needed.
DATA = paths.inputs / "datasets"

RAW = CB.resolve_dataset_root(DATA / "invoices_raw", ['batch_1'])

print(f"  invoices_raw   -> {RAW}")
print(f"                    " f"{sum(1 for _ in RAW.rglob(chr(42)) if _.is_file()):,} files")

In [ ]:
# --- Locate the real batch folders, excluding batch_3's duplicate copies ------
import pandas as pd
from PIL import Image

roots = [p for p in RAW.rglob("batch*_*") if p.is_dir() and any(p.glob("*.jpg"))]

# batch_3/batch_1/* and batch_3/batch_2/* duplicate batches 1 and 2 - drop them.
def is_dup(p: Path) -> bool:
    parts = p.parts
    return "batch_3" in parts and not any(x.startswith("batch3_") for x in parts)

leaf = sorted([p for p in roots if not is_dup(p)])
dups = sorted([p for p in roots if is_dup(p)])

print("USING these leaf folders:")
for p in leaf:
    print(f"  {p.relative_to(RAW)}: {len(list(p.glob('*.jpg')))}")
print(f"\nEXCLUDED {len(dups)} duplicate folders "
      f"({sum(len(list(p.glob('*.jpg'))) for p in dups)} images)")
print("unique images:", sum(len(list(p.glob('*.jpg'))) for p in leaf))

In [ ]:
# --- Load the batch_1 annotation CSVs = the only real ground truth ------------
ann_csvs = sorted(RAW.rglob("batch1_*.csv"))
print("annotation CSVs:", [p.name for p in ann_csvs])

gt = pd.concat([pd.read_csv(p) for p in ann_csvs], ignore_index=True)
gt["stem"] = gt["File Name"].astype(str).str.replace(".jpg", "", regex=False).str.strip()
gt = gt.drop_duplicates("stem")
annotated = set(gt["stem"])

print("columns      :", list(gt.columns))
print("annotated ids:", len(annotated))
print("\nJson Data holds invoice / items / subtotal / payment_instructions;")
print("OCRed Text holds the page-level transcription. Both are REAL ground truth.")
gt.head(2)

In [ ]:
# --- Annotation-aware sample -------------------------------------------------
TARGET = 750
SEED = 42

rows = []
for d in leaf:
    for img in sorted(d.glob("*.jpg")):
        rows.append({"document_id": img.stem, "src": img,
                     "batch": d.name, "has_gt": img.stem in annotated})
allimgs = pd.DataFrame(rows)
print("total unique images:", len(allimgs), "| with GT:", int(allimgs.has_gt.sum()))

# Take every annotated image first, then top up with unannotated ones for visual variety.
have = allimgs[allimgs.has_gt]
rest = allimgs[~allimgs.has_gt]
take_gt = have.sample(min(TARGET, len(have)), random_state=SEED)
need = TARGET - len(take_gt)
sample = pd.concat([take_gt,
                    rest.groupby("batch", group_keys=False)
                        .apply(lambda g: g.sample(max(1, need // rest.batch.nunique()),
                                                  random_state=SEED))
                    ][: 2 if need > 0 else 1]).head(TARGET).reset_index(drop=True)

cov = sample.has_gt.mean()
print(f"sampled {len(sample)} | GT coverage {cov:.1%}  (was 26.3%)")
assert sample.document_id.is_unique, "document_id collision - check the duplicate filter"
sample.batch.value_counts()

In [ ]:
# --- Build the contracted manifest -------------------------------------------
from tqdm.auto import tqdm

STAGE = Path("/content/out"); (STAGE / "images").mkdir(parents=True, exist_ok=True)
recs = []
for r in tqdm(sample.itertuples(), total=len(sample)):
    try:
        with Image.open(r.src) as im:
            w, h = im.size
        corrupt = False
    except Exception:
        w = h = 0; corrupt = True
    dst = STAGE / "images" / f"{r.document_id}.jpg"
    shutil.copyfile(r.src, dst)
    recs.append({"document_id": r.document_id, "image_path": f"inputs/images/{dst.name}",
                 "width": w, "height": h, "file_type": "jpg",
                 "is_corrupt": corrupt, "split": None, "has_ground_truth": r.has_gt})

man = pd.DataFrame(recs)

# Stratify the split by GT availability so the test set is not starved of labels.
import numpy as np
rng = np.random.default_rng(SEED)
man["split"] = "train"
for flag in [True, False]:
    idx = man.index[man.has_ground_truth == flag].to_numpy()
    rng.shuffle(idx)
    n = len(idx)
    man.loc[idx[: int(.16 * n)], "split"] = "val"
    man.loc[idx[int(.16 * n): int(.30 * n)], "split"] = "test"

man.to_csv(STAGE / "invoice_manifest.csv", index=False)
print(man.split.value_counts().to_dict())
print("GT coverage per split:")
print(man.groupby("split").has_ground_truth.mean().round(3))

In [ ]:
# --- Figures + data-quality report -------------------------------------------
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG = Path("/content/out/figures"); FIG.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(3, 4, figsize=(14, 10))
for a, r in zip(ax.ravel(), man.sample(12, random_state=SEED).itertuples()):
    a.imshow(Image.open(STAGE / "images" / f"{r.document_id}.jpg")); a.axis("off")
    a.set_title(f"{r.document_id}\n{'GT' if r.has_ground_truth else 'no GT'}", fontsize=8)
fig.suptitle("Sample of the annotation-aware invoice manifest", fontsize=13)
fig.tight_layout(); fig.savefig(FIG / "sample_invoice_grid.png", dpi=150); plt.close(fig)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
man.split.value_counts().plot.bar(ax=ax[0], title="split sizes", rot=0)
man.groupby("split").has_ground_truth.mean().plot.bar(ax=ax[1], title="GT coverage by split", rot=0)
ax[1].set_ylim(0, 1)
man.plot.scatter("width", "height", s=6, alpha=.3, ax=ax[2], title="image dimensions")
fig.tight_layout(); fig.savefig(FIG / "preprocessing_examples.png", dpi=150); plt.close(fig)

REP = Path("/content/out/data_quality_report.md")
REP.write_text(f'''# Data Quality Report — Rolando (Colab, annotation-aware)

- Unique images available: {len(allimgs)} (duplicate batch_3 copies excluded)
- Sampled: {len(man)}
- **Ground-truth coverage: {cov:.1%}** (previous local run: 26.3%)
- Split: {man.split.value_counts().to_dict()}
- Corrupt images: {int(man.is_corrupt.sum())}
- Annotated ids available: {len(annotated)}

## Known limitations
- Annotation CSVs exist only for batch_1, so high coverage is achieved by *preferring* batch_1
  images. This trades some cross-batch visual diversity for label availability — state this
  in the report.
- `batch_3/` duplicates batches 1 and 2; those copies are excluded to avoid train/test leakage.
''', encoding="utf-8")
print(REP.read_text()[:600])

In [ ]:
# --- Provenance: the _run block makes cross-run model comparison possible ------
def run_block(**kw):
    """Stamp every metrics JSON with how it was produced, so local-CPU and Colab-GPU
    results can be charted against each other later."""
    b = {
        "profile": P.get("profile_name", "colab_gpu"),
        "device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"),
        "epochs": P.get("epochs"), "imgsz": P.get("imgsz"), "batch": P.get("batch"),
        "wall_clock_sec": round(time.time() - T0, 1),
        "timestamp_utc": RUN_TS, "member": "rolando",
    }
    b.update(kw)
    return b


In [ ]:
# --- metrics + publish --------------------------------------------------------
met = Path("/content/out/ingestion_metrics.json")
met.write_text(json.dumps({
    "n_images": len(man), "gt_coverage": round(float(cov), 4),
    "split": man.split.value_counts().to_dict(),
    "unique_available": len(allimgs), "annotated_available": len(annotated),
    "_run": run_block(n_train_images=len(man), model="n/a (ingestion)"),
}, indent=2), encoding="utf-8")
print(met.read_text())

In [ ]:
# --- Publish to Drive (latest + immutable archive) -----------------------------
# Also copy the manifest + images into inputs/ so downstream notebooks see them.
to_publish = [
    ("predictions", STAGE / "invoice_manifest.csv"),
    ("metrics", met),
    ("figures", FIG / "sample_invoice_grid.png"),
    ("logs", REP),
]
for kind, src in to_publish:
    if src is None:
        continue
    p = Path(src)
    if not p.exists():
        print(f"  skip (not produced): {p}")
        continue
    CB.publish("rolando", p, kind, paths=paths, run_timestamp=RUN_TS)

print("\nLatest ->", paths.outputs("rolando"))
print("Archive ->", paths.run_dir("rolando", timestamp=RUN_TS))

In [ ]:
# --- Refresh inputs/ for every downstream member -------------------------------
# Rolando is the only member who writes into inputs/ - the others only read it.
shutil.copyfile(STAGE / "invoice_manifest.csv", paths.inputs / "invoice_manifest.csv")
(paths.inputs / "images").mkdir(exist_ok=True)
for p in (STAGE / "images").glob("*.jpg"):
    shutil.copyfile(p, paths.inputs / "images" / p.name)
CB.publish("rolando", FIG / "preprocessing_examples.png", "figures",
           paths=paths, run_timestamp=RUN_TS)
print("inputs/ refreshed:", len(list((paths.inputs / 'images').glob('*.jpg'))), "images")

## Report log — fill this in before you finish

Copy your answers into `presentation/member_reports/rolando_report_log.md` in the repo (or paste
them to the integrator). This is the raw material for the group report and slide deck, so be
specific and **honest about what didn't work**.

1. Why annotation-aware sampling, and what diversity did you trade away for coverage?
2. The duplicate-`batch_3` discovery — how would it have leaked into train/test?
3. Final GT coverage per split, and whether the test split has enough labels to report on.
4. Anything odd in the images (corrupt files, wild aspect ratios, rotations).

Also note anything the next stage needs from you, and which figure you'd put on a slide.